**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Scaling Neural Networks

The [ANN](../Intro_ANN/Intro_ANN.ipynb) and [CNN](../Intro_CNN/Intro_CNN.ipynb) workshops trained models with thousands of parameters. Frontier models have *billions* — and the jump is not "the same but bigger": it changes what limits you (memory and data movement, not ideas), what you measure (throughput, utilization), and even what to expect (scaling laws). This workshop builds that systems mindset with experiments you can run on a laptop CPU.

> ℹ️ All benchmarks below run on **CPU** and demonstrate the *reasoning*; sections that only make sense on GPUs are clearly marked *illustrative — not executed here*.

## 0. Introduction

Three questions organize everything:

1. **Where do the parameters and FLOPs go?** (accounting)
2. **What limits my throughput?** (compute vs memory bandwidth — the [GPU workshop's](../../Intro_GPU/Intro_GPU.ipynb) CGMA ratio, at training scale)
3. **What does more compute buy?** (scaling laws)

## 1. Pre-requisites

- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) and [Intro to ANN](../Intro_ANN/Intro_ANN.ipynb).
- [Intro to GPU Systems](../../Intro_GPU/README.md) — the memory-latency worldview.
- `pip install torch` (CPU build is fine here).

In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
print(torch.__version__, "| threads:", torch.get_num_threads())

2.13.0+cpu | threads: 24


---
### 🕐 Session 1 of 2 — *Why Scale? Accounting & Scaling Laws* (~35 min)
**Goal:** count parameters and FLOPs; see diminishing-but-predictable returns in a toy scaling study.
**Builds on:** [ANN](../Intro_ANN/Intro_ANN.ipynb). &nbsp; **Feeds into:** Session 2 (making training fast).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Why Scale? Accounting & Scaling Laws</b></summary>

**Timing (~35 min).** 12 min parameter and FLOP accounting · 8 min the $6ND$ rule and what it explains · 15 min the toy scaling study and its limits.

**Open by naming the shift in mindset, because that is what the workshop is actually teaching.** The [ANN](../Intro_ANN/Intro_ANN.ipynb) and [CNN](../Intro_CNN/Intro_CNN.ipynb) workshops asked "does this model learn?" At scale the question changes to "what is my binding constraint?" — and the answer is almost never ideas. It is memory, data movement, and wall-clock. Students who carry the small-model mindset into a large-model setting optimise the wrong thing for weeks.

**Teach counting before optimising, and insist it is a skill.** A linear layer $d_{\text{in}} \to d_{\text{out}}$ stores $d_{\text{in}}d_{\text{out}} + d_{\text{out}}$ weights and spends about $2d_{\text{in}}d_{\text{out}}$ FLOPs per sample — one multiply and one add per weight. That is the whole accounting rule, and it is enough to predict cost for most architectures. A student who can count is not at the mercy of a benchmark.

**Then derive the $6ND$ rule at the board, because it explains every headline about GPU-months.** Forward is $\approx 2$ FLOPs per parameter per token. The backward pass computes gradients with respect to both inputs and weights, so it costs about twice the forward. Total $\approx 6 \times$ params $\times$ tokens. Work an example live: 7B parameters on 1T tokens is $4\times10^{22}$ FLOPs, which at 40% utilisation of an A100 (~$300$ TFLOP/s) is roughly $4\times10^5$ GPU-hours. **One line of arithmetic converts a press release into a budget**, and rooms find that genuinely empowering.

**Make the quadratic-in-width point explicit, since it is the first real design consequence.** Doubling width quadruples parameters *and* FLOPs in the hidden layers, because the weight matrix grows in both dimensions. Depth, by contrast, is linear. That asymmetry is why "make it wider" gets expensive fast and why architecture papers argue about aspect ratio at all.

**Set up the scaling study honestly as a *shape* demonstration, not a scaling law.** We fix a task, sweep width at a fixed step budget, and plot loss against parameters on log axes. The claim is only that the qualitative structure — big early gains, then diminishing returns, then a floor — is the same structure governing billion-dollar runs. Say in advance that a real scaling law sweeps model size **and** data **and** compute together; Chinchilla's central finding is that parameters and tokens should grow in step, and a fixed-data sweep cannot see that.

**Have the room predict the floor before running.** The targets carry noise of standard deviation 0.1, so the variance is exactly 0.01, and **no model of any size can do better than MSE 0.01 on this data**. That is not a limitation of the architecture or the optimiser — it is irreducible. Students who understand this stop expecting loss to go to zero and start asking what the floor of their own dataset is. It is also the honest version of "more scale always helps": scale buys you the approach to the floor, never the floor itself.

**Be prepared for the results to be blunter than the story.** Width 4 (33 parameters) already reaches MSE 0.0108, essentially the floor, and everything from width 8 to 256 sits flat around 0.0105 — with width 256 slightly *worse* than width 64. There is no clean power law here, because the task is too easy and the floor arrives after two doublings. Do not paper over this; the debrief below treats it as the finding. **A demonstration that admits what it failed to show teaches more than one that overclaims.**
</details>

## 2. Parameter & FLOP Accounting

💡 **Intuition.** Before optimizing anything, learn to *count*. A linear layer $d_{in} \to d_{out}$ stores $d_{in} d_{out} + d_{out}$ weights and spends $\approx 2 \, d_{in} d_{out}$ FLOPs per input (one multiply + one add per weight). Rule of thumb for training: **forward ≈ 2 FLOPs/param/token, backward ≈ twice the forward** — so training cost $\approx 6 \times$ params $\times$ tokens. That one line explains most headlines about GPU-months.

In [2]:
def account(model, x):
    n_params = sum(p.numel() for p in model.parameters())
    flops_fwd = 0
    for m in model.modules():
        if isinstance(m, nn.Linear):
            flops_fwd += 2 * m.in_features * m.out_features
    print(f"params: {n_params:>10,}   fwd FLOPs/sample: {flops_fwd:>12,}   train ≈ {3*flops_fwd:,} FLOPs/sample")
    return n_params

for width in [64, 256, 1024]:
    print(f"width {width:>5}: ", end="")
    account(nn.Sequential(nn.Linear(128, width), nn.ReLU(),
                          nn.Linear(width, width), nn.ReLU(),
                          nn.Linear(width, 1)), None)
print("→ doubling width ≈ 4x params and FLOPs: cost grows QUADRATICALLY in width")

width    64: params:     12,481   fwd FLOPs/sample:       24,704   train ≈ 74,112 FLOPs/sample
width   256: params:     99,073   fwd FLOPs/sample:      197,120   train ≈ 591,360 FLOPs/sample
width  1024: params:  1,182,721   fwd FLOPs/sample:    2,361,344   train ≈ 7,084,032 FLOPs/sample
→ doubling width ≈ 4x params and FLOPs: cost grows QUADRATICALLY in width


**What just happened.** Three widths, and the cost grows far faster than the width:

| width | params | fwd FLOPs/sample | ratio to previous |
|---|---|---|---|
| 64 | 12,481 | 24,704 | — |
| 256 | 99,073 | 197,120 | **8×** for 4× width |
| 1024 | 1,182,721 | 2,361,344 | **12×** for 4× width |

**The scaling is quadratic in width, and the reason is in the shapes.** The middle layer is $w \times w$, so doubling $w$ quadruples it. The input layer is $128 \times w$ and only doubles. As $w$ grows the square term dominates, which is why the ratios above climb toward the asymptotic 16× per 4× width. **Depth is linear in cost; width is quadratic** — and that asymmetry is why architecture papers argue about aspect ratio.

**Note the relationship between the two columns, because it is the accounting rule in miniature.** FLOPs/sample $\approx 2 \times$ params in every row: one multiply and one add per weight. That is the whole forward-pass rule, and it holds for any network dominated by matrix multiplies.

**The `3 ×` in the training estimate is the backward pass, and it is worth justifying rather than accepting.** Backprop computes gradients with respect to both the inputs and the weights of each layer — two matrix products where the forward pass did one — so backward costs about twice forward, and training is about **3× forward** per sample. Combined with the previous point, this gives the rule that explains every headline about training budgets:

$$\text{training FLOPs} \approx 6 \times \text{params} \times \text{tokens}$$

**Try it on a real number, because it converts a press release into arithmetic.** A 7B-parameter model on 1T tokens needs $6 \times 7\times10^9 \times 10^{12} = 4.2\times10^{22}$ FLOPs. An A100 delivers roughly $3\times10^{14}$ FLOP/s at bf16, and real training runs achieve perhaps 40% of peak — so about $3.5\times10^5$ GPU-hours, or 1,000 GPUs for two weeks. **One line of counting, and the scale of a frontier training run stops being mysterious.**

**One honest limitation of this `account` function.** It counts only `nn.Linear` layers, so activations, normalisations, and attention are invisible to it. For an MLP that is nearly the whole cost; for a transformer the attention $QK^T$ product adds a term quadratic in sequence length, which is precisely the term that dominates at long context and motivates FlashAttention and its successors. The rule of thumb is excellent, and knowing where it stops applying is part of using it well.

## 3. A Toy Scaling Study

💡 **Intuition.** The famous scaling-law plots (loss vs compute, straight lines on log-log axes) are *empirical* — but you can reproduce their shape on a laptop. Fix a task, sweep model size with an equal training budget per size, and plot final loss vs parameters on log axes. Expect: big early gains, then a steady power-law-ish slide — and eventually a floor set by the data's intrinsic noise, which **no** amount of scale removes.

In [3]:
# Task: regress y = sin(4x) + noise. The noise floor (var 0.01) is unbeatable BY DESIGN.
rng = np.random.default_rng(0)
Xd = rng.uniform(-1, 1, (4096, 1)).astype(np.float32)
yd = np.sin(4 * Xd) + 0.1 * rng.standard_normal((4096, 1)).astype(np.float32)
Xt, yt = torch.from_numpy(Xd), torch.from_numpy(yd)

def train_width(width, steps=600):
    model = nn.Sequential(nn.Linear(1, width), nn.Tanh(),
                          nn.Linear(width, width), nn.Tanh(),
                          nn.Linear(width, 1))
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(steps):
        idx = torch.randint(0, len(Xt), (256,))
        loss = ((model(Xt[idx]) - yt[idx])**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return sum(p.numel() for p in model.parameters()), ((model(Xt) - yt)**2).mean().item()

widths = [1, 2, 4, 8, 16, 64, 256]
results = [train_width(w) for w in widths]
for w, (p, l) in zip(widths, results): print(f"width {w:>4}  params {p:>6,}  final MSE {l:.4f}")

width    1  params      6  final MSE 0.2720
width    2  params     13  final MSE 0.0219
width    4  params     33  final MSE 0.0108
width    8  params     97  final MSE 0.0105
width   16  params    321  final MSE 0.0105
width   64  params  4,353  final MSE 0.0103
width  256  params 66,561  final MSE 0.0112


**What just happened.** Seven models spanning **6 to 66,561 parameters** — four orders of magnitude — and almost all of the improvement happens in the first two steps:

| width | params | MSE | verdict |
|---|---|---|---|
| 1 | 6 | 0.2720 | underfits badly |
| 2 | 13 | 0.0219 | **12× better for 7 more parameters** |
| 4 | 33 | 0.0108 | essentially at the floor |
| 8–64 | 97–4,353 | ~0.0105 | flat |
| 256 | 66,561 | 0.0112 | slightly *worse* |

**The last 10,000× of parameters bought nothing.** From width 4 to width 256 the model grew 2,000-fold and the loss moved by 0.0003 — noise. That is the honest headline of this cell, and it is more instructive than a clean power law would be.

**The reason is the floor, and it was designed in.** The targets are $\sin(4x) + 0.1\varepsilon$, so the label noise has variance exactly $0.01$. **No model of any size can beat MSE 0.01 on this data** — the remaining error is not a modelling failure, it is the noise we added, and it is irreducible. Width 4 got within 8% of that bound. Everything after is a model competing for a prize that does not exist.

**Be clear about what this demo does and does not show.** It shows the *shape*: steep early gains, then diminishing returns, then a floor. It does **not** show a scaling law. A power law needs many decades of clean descent, and here the descent lasts two points before the floor arrives. Calling this "a laptop-scale scaling curve" is generous; calling it "a demonstration that returns diminish onto an irreducible floor" is exact. The distinction matters, because real scaling laws are an empirical claim about a *regime*, and this task never enters that regime.

**Note why width 256 is slightly worse, since it looks like a bug and is not.** With 66,561 parameters and only 600 Adam steps at lr $10^{-2}$, the largest model has not converged as tightly as the small ones — the budget is fixed in *steps*, not in progress. A larger model can also start to fit the noise in the training set, and this is training MSE with no held-out data. Either way the difference is within run-to-run variation; re-seed and the ordering of the flat region will shuffle. **Do not read structure into differences smaller than the noise.**

**And this is exactly where the real scaling literature departs from the toy.** Chinchilla's central result is that parameters and tokens must grow **together**: holding data fixed at 4,096 points and growing the model, as we did here, is precisely the mistake that produced a generation of undertrained large models. A proper sweep varies model size, data size, and compute jointly and fits a surface, not a line. What survives from this cell to that setting is the structure — power-law-ish descent onto a floor set by irreducible entropy — and nothing else.

In [4]:
ps, ls = zip(*results)
plt.figure(figsize=(7, 3))
plt.loglog(ps, ls, "o-", label="final training MSE")
plt.axhline(0.01, color="k", linestyle="--", linewidth=0.9, label="noise floor (σ²=0.01)")
plt.xlabel("parameters"); plt.ylabel("MSE"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.title("A laptop-scale scaling curve: power-law-ish descent onto the noise floor")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1901664/698132217.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** On log–log axes the curve drops steeply from 6 to 33 parameters and then runs **flat along the dashed noise floor** for the remaining three decades. The picture is a cliff followed by a plateau, not a straight line.

**Read the plateau as the honest result.** The dashed line at $\sigma^2 = 0.01$ is the variance of the label noise, and the curve reaches it at width 4 and never leaves. Everything to the right of that point is a model spending parameters on a residual that contains **no signal**. If you take one image from this workshop, take this one: *scale buys you the approach to the floor, and the floor is a property of your data.*

**Which means the first question about any scaling plan is "what is my floor?"** In this synthetic task it is known exactly because we added the noise ourselves. On real data it is unknown — the irreducible entropy of natural language, the Bayes error of an image dataset, the sensor noise in a measurement — but it exists, and a project that budgets for a loss below it has budgeted for something impossible. The plot is a demonstration of that boundary rather than of scaling.

**Note what the plot would need in order to show a scaling law, since it does not.** A power law is a *straight line on these axes over several decades*, and that requires a task hard enough that the model never saturates. Here the descent lasts two points. The famous curves in the scaling-law literature come from tasks where the floor is far below anything achieved, so the entire measured range lies on the descent. **This figure shows the endgame; those figures show the middlegame.** Both are real, and only one of them is what "scaling law" refers to.

**A methodological caveat worth flagging, because it also affects real sweeps.** The training budget is fixed at 600 steps for every model, so the largest models get the same number of updates as the smallest — but they have vastly more parameters to move. Fixed-step sweeps systematically under-train large models, which is one reason the width-256 point sits slightly above width 64. Serious scaling studies fix *compute*, not steps, and tune the learning rate per size; the difference is not cosmetic and it is exactly the methodology error that Chinchilla corrected at industrial scale.

**Finally, the structural claim that does transfer.** The qualitative shape here — steep gains, diminishing returns, an unbeatable floor — is the same shape governing runs that cost tens of millions of dollars. What changes at that scale is which regime you are operating in and how expensive it is to find out. The reasoning is identical; only the units on the axes differ.

Real scaling laws sweep **data and compute** jointly (Chinchilla's lesson: params and tokens should grow together) — but the qualitative structure you just plotted is the same one governing billion-dollar training runs.

---
### 🕐 Session 2 of 2 — *Making Training Fast* (~40 min)
**Goal:** find the actual bottleneck: batch-size throughput curves, profiling, gradient accumulation, precision.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Making Training Fast</b></summary>

**Timing (~40 min).** 12 min throughput vs batch size · 10 min profiling · 8 min gradient accumulation · 10 min the GPU toolbox as a map.

**Frame the session as one discipline applied four times: find the binding constraint, spend the cheap resource.** Batch size trades memory for overhead amortisation. Gradient accumulation trades time for memory. Mixed precision trades numerical range for bandwidth. Sharding trades communication for capacity. Every technique in this session is that same move, and students who see the pattern can reason about tools that did not exist when the workshop was written.

**Before running the throughput sweep, ask the room to predict the shape.** Most expect throughput to rise and then flatten. It does — but it also **dips**, and the dip is the interesting part. Per-sample overhead (Python, kernel launches, optimiser bookkeeping) is fixed per *step*, so larger batches amortise it; but past a point the working set no longer fits in cache (on CPU) or in memory (on GPU), and the curve turns over. **Measure the knee** rather than assuming it, because it moves with the model, the hardware, and the precision.

**Then insist on profiling before optimising, and treat the profiler table as a reading exercise.** `aten::mm` and `aten::addmm` are the matrix multiplies — actual work. Everything else is overhead. Compute the ratio live with the room: here roughly 61% of self CPU time is in matmuls, so this configuration is reasonably efficient and the lever is faster math (better BLAS, more threads, lower precision), not less Python. Had the ratio been 20%, the lever would be the opposite. **The same measurement points at two completely different fixes depending on its value**, which is exactly why guessing is expensive.

**Gradient accumulation deserves to be derived, not asserted, because students distrust it.** The gradient of a sum is the sum of gradients, so accumulating over $k$ micro-batches of equal size and scaling each loss by $1/k$ reproduces the full-batch gradient *exactly* in real arithmetic. The demo checks it numerically and gets $2\times10^{-7}$ — float32 rounding, not a discrepancy in the mathematics. Flag the two conditions that make it exact: **equal micro-batch sizes** and **scaling by the count**. Unequal sizes with naive averaging silently reweights your data, and it is a common bug.

**Say what accumulation does not buy, since the trade is asymmetric.** It gives you the *optimisation behaviour* of a large batch at the memory of a small one, paid for in wall-clock — you still run $k$ forward and backward passes. It does not make training faster; it makes a batch size possible that otherwise would not fit. That is the "spend the cheap resource" move stated precisely.

**Treat Section 7 as a map, and be explicit that nothing in it is demonstrated.** The notebook is honest about this and the room should be too. The one piece of arithmetic worth doing live is the memory budget: Adam in fp32 costs about **16 bytes per parameter** — 4 for weights, 4 for gradients, 8 for the two moment estimates — before any activations. A 7B model therefore needs ~112 GB of optimiser state alone, which exceeds any single accelerator. **That one number explains ZeRO, FSDP, sharding, and activation checkpointing all at once**, and it lands far better than a list of technique names.

**Close by connecting back to the systems workshops.** The warmup iterations in the throughput function exist because of first-touch page faults from [Intro to OS](../../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb); the compute-versus-bandwidth question is the CGMA ratio from [Intro to GPU Systems](../../Intro_GPU/Intro_GPU.ipynb) at training scale. Scaling is not a separate subject — it is systems engineering with a loss function attached.
</details>

## 4. Throughput vs Batch Size

💡 **Intuition.** Per-sample overhead (Python, kernel launches, optimizer bookkeeping) is *fixed*; compute grows with the batch. Small batches ⇒ overhead dominates and throughput climbs as batches grow; eventually arithmetic saturates the hardware and the curve flattens — or even *dips*, as very large batches spill out of cache (CPU) or run out of memory (GPU). **Measure the knee** — that's your efficient operating point.

In [5]:
model = nn.Sequential(nn.Linear(256, 512), nn.ReLU(),
                      nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

def throughput(bs, iters=30):
    x = torch.randn(bs, 256); y = torch.randint(0, 10, (bs,))
    for _ in range(3):                       # warmup (first-touch costs — remember the OS workshop!)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    tic = time.perf_counter()
    for _ in range(iters):
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    return bs * iters / (time.perf_counter() - tic)

sizes = [1, 4, 16, 64, 256, 1024]
tps = [throughput(b) for b in sizes]
for b, t in zip(sizes, tps): print(f"batch {b:>5}: {t:>10,.0f} samples/s")

batch     1:      2,678 samples/s
batch     4:     15,707 samples/s
batch    16:     52,695 samples/s
batch    64:    140,170 samples/s
batch   256:    274,296 samples/s
batch  1024:    231,011 samples/s


**What just happened.** Identical arithmetic per sample, and a **102× spread in throughput** depending only on how many samples you hand over at a time:

| batch | samples/s | vs batch 1 |
|---|---|---|
| 1 | 2,678 | 1× |
| 4 | 15,707 | 5.9× |
| 16 | 52,695 | 19.7× |
| 64 | 140,170 | 52× |
| 256 | **274,296** | **102×** |
| 1024 | 231,011 | 86× — *down* |

**The rise is overhead amortisation, and you can see the fixed cost directly.** At batch 1 a step takes 373 µs to do a few hundred thousand FLOPs — work that should take microseconds. The rest is Python dispatch, autograd graph construction, kernel launches, and optimiser bookkeeping, all charged **once per step regardless of batch size**. Quadruple the batch and that fixed cost is split four ways. Early in the table throughput scales almost linearly with batch size, which is the signature of a regime where **you are not computing, you are dispatching**.

**The peak at 256 is the knee, and it is the number this cell exists to find.** By then each step is doing enough arithmetic that overhead is amortised away, and the hardware — not the framework — is the limit.

**The dip at 1024 is the most instructive row, so do not skip it.** Throughput *fell* by 16%. The working set stopped fitting in cache: activations for 1024×512 floats are 2 MB per layer, and once the model's live tensors exceed L2, every matmul starts fetching from main memory. **The bottleneck changed from dispatch overhead to memory bandwidth** — the same compute-versus-bandwidth question as the CGMA ratio in [Intro to GPU Systems](../../Intro_GPU/Intro_GPU.ipynb), here on a CPU. On a GPU the analogous cliff is an out-of-memory error rather than a slowdown, which is less subtle and more annoying.

**So "bigger batches are faster" is true only up to a machine-specific point, and the point must be measured.** It moves with the model size, the hardware, the thread count, and the precision. The numbers above are for 24 CPU threads on this particular machine and will not reproduce elsewhere — the *shape* will.

**Two details in the benchmark that are not decoration.** The three warmup iterations exist because the first touch of each page triggers a fault and allocation — first-touch behaviour straight out of [Intro to OS](../../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — and lazy library initialisation happens on the first call too. Time those and you measure setup, not steady state. And `time.perf_counter` is the right clock: monotonic and high-resolution, unlike `time.time`.

**One caveat about what this measures.** Throughput is samples per second, not progress per second. A batch of 1024 does one optimiser step where 4 batches of 256 do four; if the larger batch needs proportionally more steps to converge, its throughput advantage evaporates. **Samples/s is a systems metric and convergence is a learning metric**, and optimising the first without watching the second is a classic way to make training slower while making the dashboard look better.

In [6]:
plt.figure(figsize=(7, 2.8))
plt.semilogx(sizes, tps, "o-")
plt.xlabel("batch size"); plt.ylabel("samples/s"); plt.grid(True, which="both", alpha=0.3)
plt.title("Throughput vs batch size: find the knee (numbers vary by machine)")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1901664/1497128088.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The curve climbs steeply, bends over, peaks at batch 256, and then **turns down**. Three regimes in one plot, and each has a different bottleneck and a different fix.

**Left of the knee — overhead-bound.** Throughput rises almost proportionally with batch size, because the fixed per-step cost (Python dispatch, autograd bookkeeping, kernel launches, optimiser update) is being split across more samples. Anything you do to the math here is wasted effort; the fix is *fewer, bigger steps*, or a compiled graph that removes the dispatch entirely.

**At the knee — compute-bound, which is where you want to be.** The arithmetic finally dominates, and the hardware rather than the framework sets the pace. This is the efficient operating point, and finding it is the entire purpose of running the sweep.

**Right of the knee — memory-bound.** The working set has outgrown cache, so the matmuls stall waiting on main memory and throughput *falls*. More batch now costs you. On a GPU the same boundary usually announces itself as an out-of-memory error instead of a gentle decline — cruder, but at least unambiguous.

**The one instruction to take away: measure, do not assume.** The knee here is at 256 on 24 CPU threads with this particular model. Change the width, the precision, the thread count, or the machine and it moves. There is no default batch size; there is a sweep you run once and a number you then use. **The shape of this curve is universal; the location of its peak never is.**

**Note that the semilog axis is doing real work in how you read it.** Batch sizes span 1 to 1024, so on a linear axis the first four points would collapse against the origin and the near-linear overhead-bound regime would be invisible. Choosing the axis so that the *interesting* structure is legible is a small skill worth naming — the same reason the scaling plot earlier used log–log.

**And one thing the plot cannot tell you.** It shows samples per second, not loss per second. Large batches do fewer optimiser steps for the same data, and if convergence needs a roughly fixed number of *steps*, then the fastest configuration by this metric can be the slowest by wall-clock-to-target-loss. **Always pair a throughput sweep with a convergence check**; otherwise you have optimised the dashboard rather than the training run.

## 5. Profile Before You Optimize

Guessing at bottlenecks is how you optimize the wrong thing. PyTorch ships a profiler — read its table before touching your code.

In [7]:
from torch.profiler import profile, ProfilerActivity

x = torch.randn(256, 256); y = torch.randint(0, 10, (256,))
with profile(activities=[ProfilerActivity.CPU]) as prof:
    for _ in range(10):
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=8))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
    autograd::engine::evaluate_function: AddmmBackward0         0.84%      96.957us        42.82%       4.972ms     165.720us            30  
                                         AddmmBackward0         0.63%      72.864us        40.29%       4.678ms     155.941us            30  
                                               aten::mm        38.70%       4.493ms        38.76%       4.500ms      89.990us            50  
                                           aten::linear         2.09%     243.018us        28.48%       3.307ms     110.236us            30  
      

USDT:2026-07-22 12:32:19 1901664:1901664 SyncActivityProfilerHandler.cpp:52] profiler_start
USDT:2026-07-22 12:32:19 1901664:1901664 SyncActivityProfilerHandler.cpp:59] profiler_stop


**What just happened.** 11.6 ms of CPU time for 10 training steps at batch 256, broken down by operation. The table is sorted by total time, but the column that answers the question is **Self CPU** — time spent *in* an operation rather than in its children, so the percentages sum to 100% without double-counting.

**Add up the real work first.** `aten::mm` is 38.7% and `aten::addmm` is 22.5%, so the matrix multiplies account for **about 61% of self CPU time**. Everything else — dispatch, autograd graph traversal, the optimiser's `add_` calls, ReLU — is overhead. That single ratio is what the profiler is for.

**Now read the ratio as a decision, because that is the skill.** At 61% math this configuration is reasonably efficient, and the lever is *faster arithmetic*: more threads, a better BLAS, lower precision, tensor cores. Had the ratio come back at 20%, the same table would point at the opposite fix — reduce Python and dispatch overhead via `torch.compile`, CUDA graphs, or simply a larger batch. **The identical measurement recommends contradictory actions depending on its value**, which is exactly why guessing at a bottleneck is expensive and why "profile before you optimise" is not a slogan.

**Two rows repay a second look.** `AddmmBackward0` at 42.8% CPU-total against `aten::linear` at 28.5% is the backward pass costing roughly 1.5× the forward here — close to the theoretical 2× from Session 1's $6ND$ rule, with the shortfall explained by the bias gradient being cheap. And `aten::add_` appears **60 times** for 10 steps: six parameter tensors updated per SGD step. With Adam that row would be several times larger, because each parameter needs two moment updates as well.

**Sanity-check the profiler against the sweep, since agreement between two independent measurements is worth more than either alone.** 11.6 ms for 10 steps of batch 256 is 1.16 ms per step, or about 220,000 samples/s — against 274,000 measured in the throughput cell. The gap is profiler overhead, which is real and always inflates the numbers. **A profiler tells you where time goes, not how much time there is**; use it for proportions and a plain timer for absolutes.

**One structural caveat.** This is a CPU profile. On a GPU you must add `ProfilerActivity.CUDA` and synchronise, because kernel launches are asynchronous — the CPU returns immediately and a naive CPU-only profile will report that your model spends all its time in `cudaLaunchKernel` and none in the actual computation. That confusion is nearly universal on first contact with GPU profiling, and it is worth knowing before you meet it.

Reading it: `addmm`/`mm` rows are your matrix multiplies (real work); everything else is overhead. The ratio between them tells you whether to seek faster math or less overhead.

## 6. Gradient Accumulation

💡 **Intuition.** Want the optimization behavior of batch 1024 but only memory for 256? Run 4 micro-batches, **add up their gradients, step once**. Gradients are sums over samples, so the result is mathematically the large batch (identical when the loss averages per micro-batch of equal size and you scale by the count). This is the standard trick behind every 'effective batch size' line in a paper.

In [8]:
def grads_of(batches, accumulate):
    torch.manual_seed(7)
    m = nn.Linear(8, 1)
    if accumulate:
        m.zero_grad()
        for xb, yb in batches:
            (((m(xb) - yb)**2).mean() / len(batches)).backward()   # scale by micro-batch count
    else:
        xb = torch.cat([b[0] for b in batches]); yb = torch.cat([b[1] for b in batches])
        ((m(xb) - yb)**2).mean().backward()
    return m.weight.grad.clone()

data = [(torch.randn(4, 8), torch.randn(4, 1)) for _ in range(4)]
g_acc, g_big = grads_of(data, True), grads_of(data, False)
print("max |accumulated − full-batch gradient|:", (g_acc - g_big).abs().max().item())
assert torch.allclose(g_acc, g_big, atol=1e-6)
print("identical — accumulation IS the big batch, paid for in time instead of memory")

max |accumulated − full-batch gradient|: 2.384185791015625e-07
identical — accumulation IS the big batch, paid for in time instead of memory


**What just happened.** Four micro-batches of 4, accumulated, versus one batch of 16 in a single pass. The largest disagreement anywhere in the gradient is $2.4 \times 10^{-7}$, and the `assert` at `atol=1e-6` passes. **Accumulation reproduces the large-batch gradient.**

**Note the precise claim, because "identical" is a shade stronger than what was measured.** $2.4\times10^{-7}$ is not zero — it is float32 rounding, right at the $\varepsilon_{\text{mach}} \approx 1.2\times10^{-7}$ scale for gradient entries of order 1. The two computations are **mathematically identical and numerically indistinguishable in float32**; summing in a different order gives different rounding, and that is all this residual is. In float64 it would drop to $\sim10^{-16}$; in bf16 it would be far larger, which matters when you accumulate over dozens of micro-batches in mixed precision.

**Why it works is one line, and it is worth putting on the board.** The loss is a sum over samples, differentiation is linear, so the gradient of a sum is the sum of gradients:

$$\nabla\Big(\tfrac{1}{N}\sum_{i=1}^{N} \ell_i\Big) = \sum_{k=1}^{K} \nabla\Big(\tfrac{1}{K}\cdot\tfrac{1}{N/K}\sum_{i \in \text{batch}_k} \ell_i\Big)$$

PyTorch **accumulates** into `.grad` by default rather than overwriting — the behaviour that makes `opt.zero_grad()` necessary and that beginners regard as a design wart. Here it is exactly the feature being exploited.

**The two conditions that make it exact are both visible in the code, and both are common bugs when they are not.** The micro-batches must be **equal in size**, and each loss must be **scaled by `1 / len(batches)`**. Omit the scaling and your gradient is $K$ times too large — which usually presents as mysterious divergence after a refactor. Use unequal micro-batches with per-batch `.mean()` and you have silently upweighted the samples in the smaller ones. The final ragged batch of a dataloader is precisely where this bites.

**Be precise about what the trick buys, since the trade is asymmetric.** You get the *optimisation behaviour* of batch 16 at the *memory* of batch 4. You do **not** get any speedup — four forward and backward passes still run, plus the fixed per-step overhead the throughput sweep just measured, so accumulation is if anything slightly slower. **It converts a memory constraint into a time cost**, which is the Session 2 discipline in its purest form: find the binding constraint, spend the cheap resource.

**And it is the reason "effective batch size" appears in every paper's method section.** A model too large to fit batch 512 on one device trains with micro-batch 32 and 16 accumulation steps and reports 512, because that is genuinely the batch the optimiser saw. Extend the same idea across machines — accumulate locally, all-reduce the gradients — and you have `DistributedDataParallel`. **Data parallelism is gradient accumulation with a network in the middle**, which is why this six-line demo is worth as much attention as the section it sits in.

## 7. The GPU-Scale Toolbox *(illustrative — not executed in this CPU notebook)*

On real accelerators the same reasoning continues with hardware-specific tools — presented here as a map, since none of this can be demonstrated honestly on CPU:

- **Mixed precision** (`torch.autocast` + fp16/bf16): tensor cores double-to-quadruple matmul throughput and halve activation memory; loss scaling guards small fp16 gradients.
- **Data parallelism** (`DistributedDataParallel`): replicate the model, split the batch, all-reduce gradients — gradient accumulation across machines, plus a network.
- **Memory arithmetic**: Adam training in fp32 costs ≈ 16 bytes/param (weights 4 + grads 4 + two moments 8) before activations — a 7B model wants ~112 GB, hence sharding (ZeRO/FSDP), activation checkpointing (recompute instead of store), and model/pipeline parallelism.

Each is the Session-2 mindset — *find the binding constraint, spend the cheap resource* — applied to a bigger machine.

## 8. Conclusion

Scaling is accounting plus bottleneck-hunting: count params and FLOPs, measure the throughput knee, profile before optimizing, accumulate when memory binds, and expect power-law returns onto a noise floor. The mindset transfers unchanged from laptop to cluster.

---
## Where next

- [Intro to GPU Systems](../../Intro_GPU/README.md) — the memory hierarchy all of this optimizes against.
- [Intro to Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — the architecture the scaling laws were measured on.
- [Intro to OS](../../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — first-touch, scheduling, and why your warmup iterations exist.